In [ ]:
#!/usr/bin/env python3
import numpy as np
import xarray as xr
from pathlib import Path

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# BATSIRAI : nt: int = 9, t0: str = "2022-02-01T00:00:00.000000000", dt_hours: int = 6,
# BELNA : nt: int = 9, t0: str = "2019-12-06T00:00:00.000000000", dt_hours: int = 6,
# IANOS : nt: int = 4, t0: str = "2020-09-16T18:00:00.000000000'", dt_hours: int =18 ,
# CHIDO : nt: int = 5, t0: str = "2024-12-12T01:00:00.000000000", dt_hours: int = 1,


def make_test_prescribed_track(
    outfile: str = "test_prescribed_track_curved_batsirai.nc",
    nt: int = 9, t0: str = "2022-02-01T00:00:00.000000000", dt_hours: int = 6,
    lat_start: float = -17,
    lon_start: float = 62.0,
    lat_end: float = -19,
    lon_end: float = 56,
    amp_offset: float = 1.0,  # amplitude maximale du décalage (en degrés)
) -> None:
    """
    Crée un NetCDF de test pour PrescribedTracker avec une trajectoire
    globalement linéaire mais spatialement courbée, puis la trace sur une carte.
    """

    # Axe temporel régulier
    t0_np = np.datetime64(t0)
    time = t0_np + np.arange(nt) * np.timedelta64(dt_hours, "h")

    # Paramètre de progression de 0 à 1
    s = np.linspace(0.0, 1.0, nt)

    # Partie linéaire
    lat_lin = lat_start + s * (lat_end - lat_start)
    lon_lin = lon_start + s * (lon_end - lon_start)

    # Vecteur direction du segment
    dlat = lat_end - lat_start
    dlon = lon_end - lon_start
    norm = np.hypot(dlat, dlon)

    if norm == 0.0:
        lat_offset = np.zeros_like(s)
        lon_offset = np.zeros_like(s)
    else:
        # Vecteur unitaire perpendiculaire au segment
        perp_lat = -dlon / norm
        perp_lon = dlat / norm

        # Profil de courbure: 0 aux extrémités, max au milieu
        curvature = 4.0 * s * (1.0 - s)

        lat_offset = amp_offset * curvature * perp_lat
        lon_offset = amp_offset * curvature * perp_lon

    latitude = lat_lin + lat_offset
    longitude = lon_lin + lon_offset

    ds = xr.Dataset(
        data_vars={
            "latitude": ("time", latitude),
            "longitude": ("time", longitude),
        },
        coords={"time": ("time", time)},
        attrs={
            "title": "Test prescribed tracker trajectory (curved)",
            "description": (
                "Synthetic curved track from (lat_start, lon_start) "
                "to (lat_end, lon_end) with perpendicular smooth offset."
            ),
        },
    )

    outfile = Path(outfile)
    outfile.parent.mkdir(parents=True, exist_ok=True)

    # Supprimer le fichier existant avant d'écrire
    if outfile.exists():
        outfile.unlink()

    ds.to_netcdf(outfile)
    print(f"Wrote test prescribed track to: {outfile}")

    # -------------------------
    # Tracé sur une carte
    # -------------------------
    proj = ccrs.PlateCarree()

    fig = plt.figure(figsize=(6, 6))
    ax = plt.axes(projection=proj)

    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.gridlines(draw_labels=True, linewidth=0.5, linestyle=":")

    # Extent avec une petite marge autour de la track
    margin = 3.0
    lon_min = float(np.min(longitude) - margin)
    lon_max = float(np.max(longitude) + margin)
    lat_min = float(np.min(latitude) - margin)
    lat_max = float(np.max(latitude) + margin)
    # ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)

    # Tracé de la trajectoire
    ax.plot(
        longitude,
        latitude,
        marker="o",
        linestyle="-",
        transform=proj,
    )

    ax.set_title("Test prescribed track (courbée)")

    plt.tight_layout()
    plt.show()
    # Si tu préfères un fichier plutôt qu'un affichage, tu peux utiliser :
    # plt.savefig("test_prescribed_track_curved.png", dpi=150)


if __name__ == "__main__":
    make_test_prescribed_track()